### Overview

Extracts survival data of TCGA-BRCA samples for survival analysis.

In [1]:
import pandas as pd
import numpy as np

In [3]:
# import tcga brca clinical data
tcga_metadata = pd.read_csv("C:/Users/User/Documents/master_thesis_project_analysis/datasets/TCGA_BRCA/TCGA_BRCA_cleaned_data/cleaned_sample info.csv",
                            header=0, index_col=0)

In [4]:
# check survival status counts
print(tcga_metadata['vital_status'].value_counts())
print(tcga_metadata['paper_vital_status'].value_counts())
print(tcga_metadata['vital_status'].tolist() == tcga_metadata['paper_vital_status'].tolist())

vital_status
Alive    888
Dead     143
Name: count, dtype: int64
paper_vital_status
Alive    888
Dead     143
Name: count, dtype: int64
True


In [6]:
tcga_metadata['paper_BRCA_Subtype_PAM50'].value_counts()

paper_BRCA_Subtype_PAM50
LumA     555
LumB     209
Basal    185
Her2      82
Name: count, dtype: int64

In [7]:
# check missing values in days to follow up and days to death
print(tcga_metadata['days_to_last_follow_up'].isna().sum())
print(tcga_metadata['days_to_death'].isna().sum())
print(tcga_metadata['paper_days_to_last_followup'].isna().sum())
print(tcga_metadata['paper_days_to_death'].isna().sum())

99
889
99
889


In [8]:
# extract days to follow up, days to death and vital status
tcga_survival = tcga_metadata.loc[:, ['vital_status', 'days_to_last_follow_up', 'days_to_death']]
tcga_survival.head(3)

,vital_status,days_to_last_follow_up,days_to_death
TCGA-D8-A146-01A-31R-A115-07,Alive,643.0,NaN
TCGA-AQ-A0Y5-01A-11R-A14M-07,Dead,NaN,172.0
TCGA-C8-A274-01A-11R-A16F-07,Alive,508.0,NaN


In [9]:
# create a column called overall_survival_days and make it same as days_to_death
tcga_survival['overall_survival_days'] = tcga_survival['days_to_death']
tcga_survival.head(3)

,vital_status,days_to_last_follow_up,days_to_death,overall_survival_days
TCGA-D8-A146-01A-31R-A115-07,Alive,643.0,NaN,NaN
TCGA-AQ-A0Y5-01A-11R-A14M-07,Dead,NaN,172.0,172.0
TCGA-C8-A274-01A-11R-A16F-07,Alive,508.0,NaN,NaN


In [10]:
# if there are missing values in overall_survival days, make it same as days_to_last_follow_up
tcga_survival.loc[tcga_survival['overall_survival_days'].isna(), 'overall_survival_days'] = tcga_survival.loc[tcga_survival['overall_survival_days'].isna(),'days_to_last_follow_up']
tcga_survival.head(5)

,vital_status,days_to_last_follow_up,days_to_death,overall_survival_days
TCGA-D8-A146-01A-31R-A115-07,Alive,643.0,NaN,643.0
TCGA-AQ-A0Y5-01A-11R-A14M-07,Dead,NaN,172.0,172.0
TCGA-C8-A274-01A-11R-A16F-07,Alive,508.0,NaN,508.0
TCGA-BH-A0BD-01A-11R-A034-07,Alive,554.0,NaN,554.0
TCGA-B6-A1KC-01B-11R-A157-07,Alive,1326.0,NaN,1326.0


In [11]:
# check the type of overall_survival_days and check if there are missing values
print(tcga_survival['overall_survival_days'].dtype)
print(tcga_survival['overall_survival_days'].isna().any())

float64
False


In [12]:
# create another column called overall_survival_years which is overall_survival_days divided by 365.25
tcga_survival['overall_survival_years'] = round(tcga_survival['overall_survival_days']/365.25,3)
tcga_survival.head(5)

,vital_status,days_to_last_follow_up,days_to_death,overall_survival_days,overall_survival_years
TCGA-D8-A146-01A-31R-A115-07,Alive,643.0,NaN,643.0,1.760
TCGA-AQ-A0Y5-01A-11R-A14M-07,Dead,NaN,172.0,172.0,0.471
TCGA-C8-A274-01A-11R-A16F-07,Alive,508.0,NaN,508.0,1.391
TCGA-BH-A0BD-01A-11R-A034-07,Alive,554.0,NaN,554.0,1.517
TCGA-B6-A1KC-01B-11R-A157-07,Alive,1326.0,NaN,1326.0,3.630


In [13]:
# check the max in overall_survival_years
tcga_survival.loc[tcga_survival['overall_survival_years'].idxmax(),:]

vital_status               Alive
days_to_last_follow_up    8605.0
days_to_death                NaN
overall_survival_days     8605.0
overall_survival_years    23.559
Name: TCGA-B6-A0RU-01A-11R-A084-07, dtype: object

In [15]:
# import tcga counts and subtype data
tcga_subtype_counts = pd.read_csv("C:/Users/User/Documents/master_thesis_project_analysis/datasets/TCGA_BRCA/TCGA_BRCA_cleaned_data/pam50genes_raw_counts_subtype_tcga_brca.csv", 
                       header=0, index_col=0)

In [16]:
# check if the indices match
tcga_subtype_counts.index.equals(tcga_survival.index)

True

In [17]:
# check if the subtypes match
tcga_subtype_counts['subtype'].tolist() == tcga_metadata['paper_BRCA_Subtype_PAM50'].tolist()

True

In [18]:
# merge survival and subtype data
tcga_survival_subtype = tcga_survival.join(tcga_subtype_counts.loc[:, 'subtype'])
tcga_survival_subtype.head(4)

,vital_status,days_to_last_follow_up,days_to_death,overall_survival_days,overall_survival_years,subtype
TCGA-D8-A146-01A-31R-A115-07,Alive,643.0,NaN,643.0,1.760,LumA
TCGA-AQ-A0Y5-01A-11R-A14M-07,Dead,NaN,172.0,172.0,0.471,LumA
TCGA-C8-A274-01A-11R-A16F-07,Alive,508.0,NaN,508.0,1.391,LumB
TCGA-BH-A0BD-01A-11R-A034-07,Alive,554.0,NaN,554.0,1.517,LumB


In [19]:
# there is a sample with a negative value for overall survival years, check how many there are
# remove this sample during survival analysis
print((tcga_survival_subtype['overall_survival_years'] < 0).sum())
print(tcga_survival_subtype.loc[tcga_survival_subtype['overall_survival_years'] < 0, :])

1
                             vital_status  days_to_last_follow_up  \
TCGA-PL-A8LV-01A-21R-A41B-07        Alive                    -7.0   

                              days_to_death  overall_survival_days  \
TCGA-PL-A8LV-01A-21R-A41B-07            NaN                   -7.0   

                              overall_survival_years subtype  
TCGA-PL-A8LV-01A-21R-A41B-07                  -0.019   Basal  


In [20]:
# # save the file
# tcga_survival_subtype.to_csv('tcga_brca_survival_subtype.csv')